<a href="https://colab.research.google.com/github/twyeh/university-physics/blob/main/rocket_sim_HTML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import IPython
from IPython.display import HTML

# 將原本的 Python/pythonGraph 邏輯轉換為 HTML5 Canvas 與 JavaScript 網頁互動版
html_code = """
<!DOCTYPE html>
<html>
<head>
    <meta charset='utf-8'>
    <title>Rocket Landing Simulator</title>
    <style>
        #simulator-container {
            text-align: center;
            font-family: Arial, sans-serif;
            background-color: #1e1e24;
            padding: 20px;
            border-radius: 10px;
            color: white;
            width: 95%;
            max-width: 1000px;
            margin: 0 auto;
        }
        canvas {
            background-color: #0b0b16;
            border: 2px solid #444;
            border-radius: 5px;
            display: block;
            margin: 10px auto;
            max-width: 100%;
            height: auto;
        }
        .controls-info {
            margin-top: 10px;
            font-size: 14px;
            color: #aaa;
        }
        button {
            background-color: #007bff;
            color: white;
            border: none;
            padding: 10px 20px;
            font-size: 16px;
            border-radius: 5px;
            cursor: pointer;
            margin-top: 10px;
        }
        button:hover {
            background-color: #0056b3;
        }
    </style>
</head>
<body>
    <div id='simulator-container'>
        <h2>CS110Z Rocket Simulator (HTML5 Web Version)</h2>
        <canvas id='simCanvas' width='900' height='400'></canvas>
        <div class='controls-info'>
            <strong>操作說明：</strong> 使用鍵盤方向鍵 <b>[左/右]</b> 控制火箭傾角，按住 <b>[上]</b> 鍵開啟主推進器（向上加速）。
        </div>
        <button id='resetBtn'>重新啟動模擬 (Reset)</button>
    </div>

    <script>
        const canvas = document.getElementById('simCanvas');
        const ctx = canvas.getContext('2d');
        const resetBtn = document.getElementById('resetBtn');

        // 遊戲狀態與常數
        const GRAVITY = 0.05;
        let rocket = {};
        let boat = {};
        let terrain = [];
        let keys = {};
        let isGameOver = false;
        let gameMessage = "";

        // 初始化模擬
        function initializeSimulation() {
            isGameOver = false;
            gameMessage = "";

            // 初始化火箭
            rocket = {
                x: 100 + Math.random() * 200,
                y: 50,
                vx: 1 + Math.random() * 2,
                vy: 0,
                angle: 0,
                thrust: false,
                fuel: 100
            };

            // 初始化小船 (降落平台)
            boat = {
                x: 600 + Math.random() * 150,
                y: 360,
                width: 100,
                height: 20,
                vx: -0.5
            };

            // 簡單生成地形 (水平面)
            terrain = [];
            for (let i = 0; i <= canvas.width; i += 50) {
                terrain.push({x: i, y: 380});
            }
        }

        // 監聽鍵盤事件
        window.addEventListener('keydown', function(e) {
            keys[e.code] = true;
            if (['ArrowUp', 'ArrowDown', 'ArrowLeft', 'ArrowRight', 'Space'].includes(e.code)) {
                e.preventDefault(); // 避免網頁捲動
            }
        });
        window.addEventListener('keyup', function(e) {
            keys[e.code] = false;
        });

        resetBtn.addEventListener('click', initializeSimulation);

        // 取得使用者輸入與更新火箭物理狀態
        function updateObjects() {
            if (isGameOver) return;

            // 1. 更新小船位置（左右緩慢移動）
            boat.x += boat.vx;
            if (boat.x < 500 || boat.x > 750) {
                boat.vx *= -1;
            }

            // 2. 獲取鍵盤輸入
            if (keys['ArrowLeft']) {
                rocket.angle -= 0.05;
            }
            if (keys['ArrowRight']) {
                rocket.angle += 0.05;
            }
            if (keys['ArrowUp'] && rocket.fuel > 0) {
                rocket.thrust = true;
                rocket.fuel -= 0.2;
                // 根據火箭角度計算推進力加速度
                rocket.vx += Math.sin(rocket.angle) * 0.12;
                rocket.vy -= Math.cos(rocket.angle) * 0.12;
            } else {
                rocket.thrust = false;
            }

            // 3. 引入重力與更新火箭位置
            rocket.vy += GRAVITY;
            rocket.x += rocket.vx;
            rocket.y += rocket.vy;

            // 4. 邊界與碰撞偵測
            // 撞擊小船 (降落成功條件)
            if (rocket.y >= boat.y - 15 && rocket.x >= boat.x && rocket.x <= boat.x + boat.width) {
                isGameOver = true;
                // 檢查是否安全降落 (速度與角度限制)
                if (Math.abs(rocket.vy) < 1.5 && Math.abs(rocket.vx) < 1.0 && Math.abs(rocket.angle) < 0.2) {
                    gameMessage = "🎉 降落成功！完美著陸！";
                } else {
                    gameMessage = "💥 降落失敗：速度過快或角度傾斜導致墜毀！";
                }
            }
            // 撞擊地面或出界
            else if (rocket.y >= 370 || rocket.x < 0 || rocket.x > canvas.width) {
                isGameOver = true;
                gameMessage = "💥 任務失敗：火箭墜毀於海面或地面！";
            }
        }

        // 繪製畫面
        function drawObjects() {
            ctx.clearRect(0, 0, canvas.width, canvas.height);

            // 繪製海洋與天空背景
            ctx.fillStyle = '#0b0b16';
            ctx.fillRect(0, 0, canvas.width, canvas.height);

            // 繪製海平面/地面
            ctx.fillStyle = '#1c3d5a';
            ctx.beginPath();
            ctx.moveTo(0, canvas.height);
            for (let p of terrain) {
                ctx.lineTo(p.x, p.y);
            }
            ctx.lineTo(canvas.width, canvas.height);
            ctx.fill();

            // 繪製小船 (Landing Pad)
            ctx.fillStyle = '#718096';
            ctx.fillRect(boat.x, boat.y, boat.width, boat.height);
            ctx.fillStyle = '#e53e3e';
            ctx.fillRect(boat.x + 10, boat.y - 3, boat.width - 20, 3); // 降落指示線

            // 繪製火箭
            ctx.save();
            ctx.translate(rocket.x, rocket.y);
            ctx.rotate(rocket.angle);

            // 火箭主體
            ctx.fillStyle = '#f7fafc';
            ctx.fillRect(-6, -15, 12, 30);
            // 火箭頭部
            ctx.fillStyle = '#e53e3e';
            ctx.beginPath();
            ctx.moveTo(-6, -15);
            ctx.lineTo(0, -27);
            ctx.lineTo(6, -15);
            ctx.fill();
            // 火箭尾翼
            ctx.fillStyle = '#4a5568';
            ctx.fillRect(-10, 5, 4, 10);
            ctx.fillRect(6, 5, 4, 10);

            // 繪製推進火焰
            if (rocket.thrust) {
                ctx.fillStyle = '#ecc94b';
                ctx.beginPath();
                ctx.moveTo(-4, 15);
                ctx.lineTo(0, 15 + Math.random() * 15);
                ctx.lineTo(4, 15);
                ctx.fill();
            }
            ctx.restore();

            // 繪製儀表板 (HUD)
            ctx.fillStyle = 'white';
            ctx.font = '14px Monaco, Courier New, monospace';
            ctx.fillText(`水平速度 (VX): ${rocket.vx.toFixed(2)}`, 20, 30);
            ctx.fillText(`垂直速度 (VY): ${rocket.vy.toFixed(2)}`, 20, 50);
            ctx.fillText(`火箭傾角: ${(rocket.angle * 180 / Math.PI).toFixed(1)}°`, 20, 70);

            // 燃料條
            ctx.fillText('FUEL:', 20, 100);
            ctx.fillStyle = '#4a5568';
            ctx.fillRect(70, 88, 100, 12);
            ctx.fillStyle = rocket.fuel > 30 ? '#48bb78' : '#f56565';
            ctx.fillRect(70, 88, rocket.fuel, 12);

            // 結束訊息顯示
            if (isGameOver) {
                ctx.fillStyle = 'rgba(0, 0, 0, 0.7)';
                ctx.fillRect(0, 0, canvas.width, canvas.height);
                ctx.fillStyle = 'white';
                ctx.font = '24px Arial';
                ctx.textAlign = 'center';
                ctx.fillText(gameMessage, canvas.width / 2, canvas.height / 2);
            }
            ctx.textAlign = 'left';
        }

        // 主循環
        function mainLoop() {
            updateObjects();
            drawObjects();
            requestAnimationFrame(mainLoop);
        }

        // 啟動模擬
        initializeSimulation();
        mainLoop();
    </script>
</body>
</html>
"""

# 在 Colab 中輸出渲染網頁
HTML(html_code)